In [ ]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA')\
    .named('template_pool').stylize('cre', style='red')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='random', 
                                        num_states=5, 
                                        style_background='goldenrod',
                                        style_mutations='yellow bold',
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style_background='salmon',
                                            style_deletion='red bold',
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_').named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              seq_name_pos_prefix='pos_', 
                                              seq_name_site_prefix='site_',
                                              style_background='blue',
                                              style_insertion='cyan bold',
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool], name='stacked_pool')\
    .repeat_states(2, seq_name_prefix='v_', op_iter_order=-2, name='repeated_pool')\
    .insert_kmers('bc', mode='random', length=5, seq_name_prefix='bc_', style_kmers='green')\
    .named('combo_pool').stylize(which='tags', style='gray')

combo_pool.print_library(show_name=True,seed=12)

pool[12]: seq_length=None, num_states=40
state  name                          seq
    0  mut_0.v_0.bc_0                TCCCGACT<cre>GAGAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>TTCTA</bc>AGATCGGA
    1  mut_0.v_1.bc_1                TCCCGACT<cre>GAGAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>AAGGT</bc>AGATCGGA
    2  mut_1.v_0.bc_2                TCCCGACT<cre>GGAAAGCGGGTAGTGAGCACACTGG</cre>ATTACGG<bc>AGCGG</bc>AGATCGGA
    3  mut_1.v_1.bc_3                TCCCGACT<cre>GGAAAGCGGGTAGTGAGCACACTGG</cre>ATTACGG<bc>ATTCG</bc>AGATCGGA
    4  mut_2.v_0.bc_4                TCCCGACT<cre>GGAAAGCGGTCAGTAAGCACATAGG</cre>ATTACGG<bc>ATTAT</bc>AGATCGGA
    5  mut_2.v_1.bc_5                TCCCGACT<cre>GGAAAGCGGTCAGTAAGCACATAGG</cre>ATTACGG<bc>CGGTT</bc>AGATCGGA
    6  mut_3.v_0.bc_6                TCCCGACT<cre>GGAAAGCGGGCAGGGAGCATACTGG</cre>ATTACGG<bc>GTGAG</bc>AGATCGGA
    7  mut_3.v_1.bc_7                TCCCGACT<cre>GGAAAGCGGGCAGGGAGCATACTGG</cre>ATTACGG<bc>CAAAT</bc>AGATCGGA
    8  mut_4.v_0.bc_8         

Pool(id=12, name='pool[12]', op='op[12]:stylize', num_states=40)

In [2]:
combo_pool.state.print_dag()

pool[12].state (o=-2, n=40)
├── combo_pool.state (o=-2, n=40)
├── op[11]:get_kmers.state (o=-2, n=40)
└── repeated_pool.state (o=-2, n=40)
    └── [op=Product]
        ├── op[10]:repeat.state (o=-2, n=2)
        └── stacked_pool.state (o=-1, n=20)
            └── [op=Stack]
                ├── mutated_pool.state (o=0, n=5)
                │   └── [op=Product]
                │       ├── op[2]:mutagenize.state (o=0, n=5)
                │       └── pool[1].state (o=0, n=1)
                │           └── template_pool.state (o=0, n=1)
                ├── deletion_pool.state (o=0, n=5)
                │   └── [op=Product]
                │       ├── pool[4].state (o=0, n=1)
                │       ├── op[3]:deletion_scan(marker_scan).state (o=0, n=5)
                │       └── pool[1].state (o=0, n=1)
                └── insertion_pool.state (o=-1, n=10)
                    └── [op=Product]
                        ├── sites_pool.state (o=-1, n=2)
                        │   └── op[6]:fr